## Calculating NINO34 from the CMIP future stuff that I need to mimic an uninitialised simulation

In [4]:
import numpy as np
import xarray as xr
import glob
import re
import matplotlib.pyplot as plt
from functools import partial

In [28]:
client.close()

In [2]:
from dask.distributed import Client
client = Client(threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 14
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40909,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40855,Total threads: 1
Dashboard: /proxy/44379/status,Memory: 4.50 GiB
Nanny: tcp://127.0.0.1:44643,


In [5]:
def dcpp_tropical_nino34(ds,coords,tos_name='tos'):
    ds = ds.sel(time=slice(None,'2035')) # Some MIROC goes past this but I don't need it
    return ds[tos_name].where((np.abs(ds[coords[1]]).compute()<=5)
                        & ((ds[coords[0]]%360).compute()<=240)
                        & ((ds[coords[0]]%360).compute()>=190)
                        ).mean(coords[2:4]).rename('nino34')
    

In [37]:
## Everything on gadi in oi10

out_dir = '/g/data/x77/jj8842/ENSO ROM/rom_decadal/indices/'

DCPP_dir = '/g/data/oi10/replicas/CMIP6/ScenarioMIP/'

filepaths = {
            #'CanESM5':'CCCma/CanESM5/ssp245/r{M}i1p2f1/Omon/{var}/gn/{version}/{filename}',
            #'EC-Earth3':'EC-Earth-Consortium/EC-Earth3/ssp245/r{M}i1p1f1/Omon/{var}/gn/{version}/{filename}',
            #'IPSL-CM6A-LR':'IPSL/IPSL-CM6A-LR/ssp245/r{M}i1p1f1/Omon/{var}/gn/{version}/{filename}',
            'MIROC6':'MIROC/MIROC6/ssp245/r{M}i1p1f1/Omon/{var}/gn/{version}/{filename}',
            }

coord_names = {
            'CanESM5':('longitude','latitude','i','j'),
            'EC-Earth3':('lon','lat','lon','lat'), # Using the regular grid version because I know it's a funky grid in some ways
            'IPSL-CM6A-LR':('nav_lon','nav_lat','x','y'),
            'MIROC6':('longitude','latitude','x','y'),
            }
var = 'tos'

for model in filepaths:
    ensemble_members = {}

    # Find number of ensemble members
    d = DCPP_dir+'/'.join(filepaths[model].split('/')[:4])
    matching_paths = glob.glob(d.format(M='*'))
    regex = re.compile(d.format(M='(.*)'))
    M = sorted(np.array(regex.findall('\n'.join(matching_paths)),'int'))

    medium_filepath_list = []
    for m in M:

        if model == 'MIROC6':
            d = DCPP_dir+'/'.join(filepaths[model].split('/'))
            matching_paths = glob.glob(d.format(var=var,M=m,version='*',filename='*2015*'))
            matching_paths = [sorted(matching_paths)[-1]]
        else:
            # Pick out a single version:
            d = DCPP_dir+'/'.join(filepaths[model].split('/')[:8])
            matching_paths = glob.glob(d.format(var=var,M=m,version='*'))
            regex = re.compile(d.format(var=var,M=m,version='(.*)'))
            v = sorted(regex.findall('\n'.join(matching_paths)))
            v = v[-1]
    
            # Go find appropriate filenames
            d = DCPP_dir+'/'.join(filepaths[model].split('/'))
            matching_paths = glob.glob(d.format(var=var,M=m,version=v,filename='*'))

        medium_filepath_list.append(sorted(matching_paths))
    
    ds = xr.open_mfdataset(medium_filepath_list,
              combine = 'nested',
              concat_dim = ['M','time'],
              parallel = True,
              chunks = {'i':-1,'j':-1,'x':-1,'y':-1,
                        'nlat':-1,'nlon':-1,'time':-1}, # xarray doesn't seem to care about overprescribing chunks
              preprocess = partial(dcpp_tropical_nino34, coords=coord_names[model]),
              ).assign_coords({'M':M}).load()

    
    ds.to_netcdf(out_dir+model+'_SSP245_nino34.nc')  
    

In [30]:
np.array(M[1:])-np.array(M[:-1])

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1])

In [31]:
len(medium_filepath_list)

50

In [32]:
for i in medium_filepath_list:
    print(len(i))

1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
0
1
1
1
1
1
1
1
1
1
0
1
1
0
1
1
1
1
1


In [16]:
d.format(var=var,M=m,version='(.*)')

'/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r5i1p1f1/Omon/tos/gn/(.*)'

'/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r1i1p1f1/Omon/tos/gn/v20200918'

In [19]:
want = '/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r1i1p1f1/Omon/tos/gn/v20200918'
for i, l in enumerate(d.format(var=var,M=m,version='(.*)')):
    if l != want[i]:
        print(i,l,want[i])
        print(want[:i])


78 5 1
/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r
98 ( v
/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r1i1p1f1/Omon/tos/gn/
99 . 2
/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r1i1p1f1/Omon/tos/gn/v
100 * 0
/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r1i1p1f1/Omon/tos/gn/v2
101 ) 2
/g/data/oi10/replicas/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp245/r1i1p1f1/Omon/tos/gn/v20
